In [0]:
#Gold fact table and summary tables
#Check the currencies first
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

spark.table(f"{CATALOG}.silver.purchase_orders") \
     .groupBy("currency").count().show()


In [0]:
#Build the fact table
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

po = spark.table(f"{CATALOG}.silver.purchase_orders").alias("po")
v  = spark.table(f"{CATALOG}.gold.dim_vendors").alias("v")
p  = spark.table(f"{CATALOG}.gold.dim_plants").alias("p")

df_fact_po = (po
    .join(v, F.col("po.vendor_id") == F.col("v.vendor_id"), "left")
    .join(p, F.col("po.plant_code") == F.col("p.plant_code"), "left")
    .select(
        F.col("po.po_id"),
        F.col("po.vendor_id"), F.col("po.vendor_name"),
        F.col("v.supply_category"), F.col("v.quality_grade"), F.col("v.risk_rating"),
        F.col("po.part_number"), F.col("po.part_description"),
        F.col("po.order_qty"), F.col("po.unit_price"), F.col("po.line_total"),
        F.col("po.currency"),
        F.col("po.order_date"), F.col("po.expected_delivery"), F.col("po.lead_time_days"),
        F.col("po.plant_code"), F.col("p.plant_name"), F.col("p.region"),
        F.col("po.status_clean").alias("status"),
        F.col("po.notes"),
        F.current_timestamp().alias("gold_updated_at"),
    )
)

df_fact_po.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.fact_purchase_orders")
print("✅ gold.fact_purchase_orders:", df_fact_po.count())

In [0]:
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

# Weights of the composite score (business assumptions, they must add up to 1.0)
W_ON_TIME, W_AUDIT, W_QUALITY, W_RISK = 0.3, 0.3, 0.2, 0.2

po_agg = (spark.table(f"{CATALOG}.silver.purchase_orders")
    .groupBy("vendor_id")
    .agg(
        F.count("po_id").alias("total_pos_current"),
        F.avg("lead_time_days").alias("avg_lead_time_current"),
        F.concat_ws(", ", F.sort_array(F.collect_set("currency"))).alias("currencies_used"),
    ).alias("po_agg")
)

risk_points = (F.when(F.col("v.risk_rating") == "Low", 100)
                .when(F.col("v.risk_rating") == "Medium", 60)
                .otherwise(30))

quality_points = 100 - F.least(F.col("v.defect_rate_ppm") / 10, F.lit(100.0))

composite = (F.col("v.on_time_delivery_pct") * W_ON_TIME
           + F.col("v.audit_score") * W_AUDIT
           + quality_points * W_QUALITY
           + risk_points * W_RISK)

df_scorecard = (spark.table(f"{CATALOG}.silver.vendors").alias("v")
    .join(po_agg, F.col("v.vendor_id") == F.col("po_agg.vendor_id"), "left")
    .select(
        F.col("v.vendor_id"), F.col("v.vendor_name"), F.col("v.country"), F.col("v.category"),
        F.col("v.on_time_delivery_pct"), F.col("v.quality_grade"),
        F.col("v.defect_rate_ppm"), F.col("v.audit_score"), F.col("v.risk_rating"),
        F.col("v.total_spend_usd").alias("historical_spend_usd"),
        F.coalesce(F.col("po_agg.total_pos_current"), F.lit(0)).alias("total_pos_current"),
        F.col("po_agg.avg_lead_time_current"),
        F.col("po_agg.currencies_used"),
        F.round(composite, 1).alias("composite_score"),
        F.current_timestamp().alias("gold_updated_at"),
    )
)

(df_scorecard.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.gold.agg_vendor_scorecard"))
print("✅ gold.agg_vendor_scorecard:", df_scorecard.count())

In [0]:
#cell for spend by currency
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

df_spend = (spark.table(f"{CATALOG}.silver.purchase_orders")
    .groupBy("vendor_id", "currency")
    .agg(
        F.count("po_id").alias("po_count"),
        F.sum("line_total").alias("total_spend"),
    )
    .join(spark.table(f"{CATALOG}.gold.dim_vendors").select("vendor_id", "vendor_name"),
          "vendor_id", "left")
    .select("vendor_id", "vendor_name", "currency", "po_count", "total_spend",
            F.current_timestamp().alias("gold_updated_at"))
)

df_spend.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.agg_vendor_spend_by_currency")
print("✅ gold.agg_vendor_spend_by_currency:", df_spend.count())
display(df_spend.orderBy("vendor_id", "currency"))

In [0]:
#Build the inventory health table
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

df_inv_health = (spark.table(f"{CATALOG}.silver.inventory")
    .groupBy("warehouse_code", "category", "abc_class")
    .agg(
        F.count("part_number").alias("num_parts"),
        F.sum("inventory_value_usd").alias("total_inventory_value"),
        F.sum("current_stock").alias("total_units"),
        F.avg("cycle_count_variance_pct").alias("avg_variance_pct"),
        F.sum(F.when(F.col("stock_status") == "CRITICAL", 1).otherwise(0)).alias("critical_items"),
        F.sum(F.when(F.col("stock_status") == "REORDER", 1).otherwise(0)).alias("reorder_items"),
        F.sum(F.when(F.col("stock_status") == "OVERSTOCK", 1).otherwise(0)).alias("overstock_items"),
        F.sum(F.when(F.col("stock_status") == "HEALTHY", 1).otherwise(0)).alias("healthy_items"),
    )
    .withColumn("health_score", F.round(F.col("healthy_items") / F.col("num_parts") * 100, 1))
    .withColumn("gold_updated_at", F.current_timestamp())
)

df_inv_health.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.agg_inventory_health")
print("✅ gold.agg_inventory_health:", df_inv_health.count())

In [0]:
#Check your work
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

# 1. Fact table
silver_po_count = spark.table(f"{CATALOG}.silver.purchase_orders").count()
fact = spark.table(f"{CATALOG}.gold.fact_purchase_orders")
print("silver POs:", silver_po_count, " fact rows:", fact.count())

# 2. Scorecard
sc = spark.table(f"{CATALOG}.gold.agg_vendor_scorecard")
print("scorecard rows:", sc.count(), " null scores:", sc.where(F.col("composite_score").isNull()).count())
sc.select("vendor_name", "total_pos_current", "currencies_used", "composite_score") \
  .orderBy(F.desc("composite_score")).show(15, truncate=False)

# 3. THE IMPORTANT ONE: totals per currency must match between Silver and Gold
silver_tot = (spark.table(f"{CATALOG}.silver.purchase_orders")
    .groupBy("currency").agg(F.sum("line_total").alias("silver_total")))
gold_tot = (spark.table(f"{CATALOG}.gold.agg_vendor_spend_by_currency")
    .groupBy("currency").agg(F.sum("total_spend").alias("gold_total")))
silver_tot.join(gold_tot, "currency", "full_outer").orderBy("currency").show()